In [1]:
import sys
from pathlib import Path

# Bootstrap — find src/ dynamically, works on any machine
sys.path.insert(0, str(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / '.env').exists()) / 'src'))

from config import *
from download_log import load_log, update_entry, print_entry, print_stale_sources
from datetime import datetime
import pandas as pd
import os
import requests
import io
import json

log = load_log()
print(f"Log loaded. Rows: {len(log)}")
print(f"PROJECT_ROOT: {PROJECT_ROOT}")

Log loaded. Rows: 27
PROJECT_ROOT: /Users/boulanger/Documents/governance-framework


## DPI Pipeline

**Source:** Database of Political Institutions (Cruz, Keefer, Scartascini — IDB)
**Access:** Automated via IDB CKAN API — no registration required
**Download instructions:** See `docs/instructions_data_maintenance.md` — DPI section

### Framework usage
| Indicator | Concept | Role |
|-----------|---------|------|
| Party fragmentation | Political settlement | Primary tier 2 |
| Government composition | Political settlement | Primary tier 2 |
| Checks and balances | Electoral process | Supplementary |

In [4]:
import requests
import io
import json
import pandas as pd
from datetime import datetime

DPI_CKAN_BASE = "https://data.iadb.org/api/3/action/package_show"

def get_latest_dpi_url():
    """
    Query IDB CKAN API to find latest DPI dataset and its CSV download URL.
    Tries recent years backwards to find the latest available version.
    """
    current_year = datetime.today().year
    for year in range(current_year - 1, current_year - 6, -1):
        dataset_id = f"the-database-of-political-institutions-dpi-{year}"
        r = requests.get(f"{DPI_CKAN_BASE}?id={dataset_id}", timeout=15)
        if r.status_code == 200:
            data = r.json()
            if data.get('success'):
                resources = data['result'].get('resources', [])
                # Find CSV version
                csv_resources = [res for res in resources if 'csv' in res.get('name','').lower()]
                if csv_resources:
                    print(f"Found DPI {year}: {csv_resources[0]['name']}")
                    return csv_resources[0]['url'], year
    return None, None

DPI_URL, DPI_YEAR = get_latest_dpi_url()

print(f"\nDownloading DPI {DPI_YEAR}...")
response = requests.get(DPI_URL, headers=BROWSER_HEADERS, timeout=60, allow_redirects=True)
print(f"Status: {response.status_code}, Size: {len(response.content)/1024/1024:.1f}MB")

dpi_raw = pd.read_csv(io.StringIO(response.text), low_memory=False)
print(f"\nShape: {dpi_raw.shape}")
print(f"Columns: {list(dpi_raw.columns[:15])}")
print(f"Years: {dpi_raw['year'].min()} — {dpi_raw['year'].max()}")
print(f"Countries: {dpi_raw['countryname'].nunique()}")

Found DPI 2023: DPI 2023 (CSV Version)

Status: 200, Size: 5.4MB

Shape: (8731, 127)
Columns: ['countryname', 'ifs', 'year', 'system', 'yrsoffc', 'finittrm', 'yrcurnt', 'multpl', 'military', 'defmin', 'percent1', 'percentl', 'prtyin', 'execme', 'execrlc']
Years: 1975-01-01 — 2023-01-01
Countries: 183


In [5]:
# Inspect key variable groups for framework relevance
print("All columns:")
for col in dpi_raw.columns:
    print(f"  {col}")

All columns:
  countryname
  ifs
  year
  system
  yrsoffc
  finittrm
  yrcurnt
  multpl
  military
  defmin
  percent1
  percentl
  prtyin
  execme
  execrlc
  execnat
  execrurl
  execreg
  execrel
  execage
  allhouse
  nonchief
  gov1me
  gov1seat
  gov1vote
  gov1rlc
  gov1nat
  gov1rurl
  gov1reg
  gov1rel
  gov1age
  gov2me
  gov2seat
  gov2vote
  gov2rlc
  gov2nat
  gov2rurl
  gov2reg
  gov2rel
  gov2age
  gov3me
  gov3seat
  gov3vote
  gov3rlc
  gov3nat
  gov3rurl
  gov3reg
  gov3rel
  gov3age
  govoth
  govothst
  govothvt
  opp1me
  opp1seat
  opp1vote
  opp1rlc
  opp1nat
  opp1rurl
  opp1reg
  opp1rel
  opp1age
  opp2me
  opp2seat
  opp2vote
  opp3me
  opp3seat
  opp3vote
  oppoth
  oppothst
  oppothvt
  ulprty
  numul
  ulvote
  oppmajh
  oppmajs
  dateleg
  dateexec
  legelec
  exelec
  liec
  eiec
  mdmh
  mdms
  ssh
  pluralty
  pr
  housesys
  sensys
  thresh
  dhondt
  cl
  select
  fraud
  auton
  muni
  state
  author
  stconst
  gwno
  termlimit
  numgov
  numvote


In [6]:
# Select framework-relevant DPI variables
# Covers Concepts 1 (political settlement), 2 (stability), 19 (legislative checks), 20 (electoral process)

KEEP_COLS = {
    # Identifiers
    'countryname': 'country_name',
    'ifs':         'ifs_code',
    'year':        'year',

    # Government system
    'system':      'dpi_govt_system',
    'military':    'dpi_military_executive',

    # Party/government fragmentation — Concept 1
    'prtyin':      'dpi_parties_in_govt',
    'numgov':      'dpi_num_govt_parties',
    'numopp':      'dpi_num_opp_parties',
    'govfrac':     'dpi_govt_fragmentation',
    'oppfrac':     'dpi_opp_fragmentation',
    'frac':        'dpi_total_fragmentation',
    'polariz':     'dpi_polarization',
    'gov1seat':    'dpi_govt1_seat_share',
    'opp1seat':    'dpi_opp1_seat_share',
    'herfgov':     'dpi_herfindahl_govt',
    'herfopp':     'dpi_herfindahl_opp',

    # Executive tenure — Concept 2
    'yrsoffc':     'dpi_exec_years_in_office',
    'yrcurnt':     'dpi_exec_years_current_term',
    'stabs':       'dpi_cabinet_stability',

    # Checks and balances — Concept 19
    'checks':      'dpi_checks_balances',
    'allhouse':    'dpi_govt_controls_all_houses',
    'oppmajh':     'dpi_opp_majority_lower',
    'oppmajs':     'dpi_opp_majority_upper',
    'maj':         'dpi_majority_govt',

    # Electoral — Concept 20
    'legelec':     'dpi_legislative_election_year',
    'exelec':      'dpi_executive_election_year',
    'liec':        'dpi_leg_electoral_competitiveness',
    'eiec':        'dpi_exec_electoral_competitiveness',
    'pluralty':    'dpi_plurality_system',
    'pr':          'dpi_pr_system',
    'fraud':       'dpi_election_fraud',
}

# Filter and rename
dpi = dpi_raw[[c for c in KEEP_COLS.keys() if c in dpi_raw.columns]].copy()
dpi = dpi.rename(columns={k: v for k, v in KEEP_COLS.items() if k in dpi.columns})

# Extract year from date string — no hardcoding
dpi['year'] = pd.to_datetime(dpi['year']).dt.year

# Filter to framework start year
dpi = dpi[dpi['year'] >= FRAMEWORK_START_YEAR].copy()
dpi = dpi.sort_values(['country_name', 'year']).reset_index(drop=True)

print(f"Shape: {dpi.shape}")
print(f"Years: {dpi['year'].min()} — {dpi['year'].max()}")
print(f"Countries: {dpi['country_name'].nunique()}")
print(f"\nMissing values (%) — top 10:")
missing_pct = (dpi.isnull().sum() / len(dpi) * 100).round(1)
print(missing_pct[missing_pct > 0].sort_values(ascending=False).head(10))

Shape: (6061, 31)
Years: 1990 — 2023
Countries: 182

Missing values (%) — top 10:
dpi_opp_fragmentation      24.1
dpi_herfindahl_opp         23.8
dpi_polarization           17.6
dpi_majority_govt          11.1
dpi_total_fragmentation    10.5
dpi_govt_fragmentation      9.3
dpi_herfindahl_govt         9.3
dpi_checks_balances         5.4
dpi_cabinet_stability       4.4
dpi_plurality_system        3.2
dtype: float64


In [7]:
# Save to processed
output_path = os.path.join(PROCESSED_DIR, "dpi_clean.csv")
dpi.to_csv(output_path, index=False)
print(f"Written: {output_path}")
print(f"Shape: {dpi.shape}")

# Derive data currency from data — no hardcoding
latest_year = str(int(dpi['year'].max()))

# Update download log
update_entry(
    "DPI",
    last_successful_download_date=datetime.today().strftime("%Y-%m-%d"),
    data_as_of_date=latest_year,
    local_filename="dpi_clean.csv",
    latest_available_version=f"DPI{DPI_YEAR}",
    notes=f"Automated via IDB CKAN API. Auto-detects latest version by year. "
          f"31 variables covering government composition, fragmentation, checks/balances, electoral system. "
          f"Coverage: 1975-{latest_year}, 182 countries."
)
print_entry("DPI")

Written: /Users/boulanger/Documents/governance-framework/data/processed/dpi_clean.csv
Shape: (6061, 31)
[download_log] Updated entry for DPI
  source_id: DPI
  last_attempted_date: 2026-06-12
  last_successful_download_date: 2026-06-12
  data_as_of_date: 2023
  local_filename: dpi_clean.csv
  latest_available_version: DPI2023
  no_update_reason: nan
  notes: Automated via IDB CKAN API. Auto-detects latest version by year. 31 variables covering government composition, fragmentation, checks/balances, electoral system. Coverage: 1975-2023, 182 countries.
